# 09.4 文本检索音频：把查询和曲库放进同一个嵌入空间

前面几节使用音频作为查询，本节改用文字。例如，输入“二胡独奏”，系统需要把符合描述的音频片段排在前面。文本与音频没有共同时间轴，因此逐帧对齐不再适用。这类任务称为跨模态检索（cross-modal retrieval）。

第六章介绍过 CLAP 的双编码器结构：音频编码器和文本编码器把两种输入映射到共享嵌入空间，对比学习目标拉近配对音频与文本的表示，并拉远训练批次内的不配对表示。
检索时，曲库片段和查询文本分别编码成向量，再按余弦相似度排序。模型表达的相关性不仅受训练配对影响，还取决于编码器结构、训练目标、音频预处理和查询措辞。

本 Notebook 用粗粒度类别、细粒度类别和速度属性三组查询作为诊断性对照，观察当前检查点的排序行为。

## 0. 环境与语料

本 Notebook 使用独立环境 `CODE/venv_ch09_clap`，其中 `laion-clap` 版本为 1.1.7，其他依赖按第六章的固定版本复刻。CLAP 权重直接读取第六章已下载的检查点，不在运行时重新下载。

曲库含 190 个 5 秒片段，来自 CTIS 中国传统乐器声音、GTZAN 十流派、MUSDB18-HQ 分轨，以及三种音色的民歌合成渲染。CTIS 部分包括五类胡琴和唢呐、琵琶等六类其他乐器。所有片段统一为 48 kHz、单声道、5 秒并做峰值归一，以减少采样率、声道、时长和整体幅度造成的格式混杂。不同来源的录音条件、曲目内容和类别构成仍可能影响结果，这些混杂不会随预处理消失。清单还为每条片段记录一条按固定模板生成的英文描述，用于检查类别与查询措辞。


In [ ]:
from pathlib import Path
import os
import sys
import time

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")  # laion_clap 文本塔初始化要加载 roberta tokenizer

# 路径推断：从 cwd 向上找含 CODE/chapter09/_common 的目录；ROOT 指向 CODE/chapter09/
_p = Path.cwd()
while not (_p / "CODE" / "chapter09" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter09/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter09"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from _common.env_check import check_notebook_env
from _common.paths import portable_path
from _common.plotting import finish_figure, setup_plot_style

check_notebook_env("09_4_clap_retrieval")

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

CKPT = ROOT.parent / "chapter06" / "pretrained_transformer" / "outputs" / "checkpoints" / "music_audioset_epoch_15_esc_90.14.pt"
CORPUS_DIR = ROOT.parent / "datasets" / "clap_corpus_ch09"
OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

if not CKPT.exists():
    raise FileNotFoundError(f"CLAP checkpoint 缺失:{rel(CKPT)};沿用第六章下载流程补齐后重跑")

import laion_clap

t0 = time.perf_counter()
model = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-base")
model.load_ckpt(str(CKPT))
model.eval()
print(f"CLAP 加载 {time.perf_counter() - t0:.1f} 秒")

corpus = pd.read_csv(CORPUS_DIR / "manifest.csv")
print(f"语料 {len(corpus)} 片段")
print(corpus.groupby("source_group").size().to_string())


In [ ]:
SR = 48000
BATCH = 16

# 全部片段编码（读取时已归一为 48 kHz/5 s，直接按批送进音频塔）
def load_clip(path):
    # 语料已被 prepare 脚本归一为 48 kHz/240000 采样；截补是防御性的，正常不触发
    y, _ = librosa.load(path, sr=SR, mono=True)
    target = SR * 5
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target]

t0 = time.perf_counter()
audio_embs = []
paths = [CORPUS_DIR / p for p in corpus["wav_path"]]
for k in range(0, len(paths), BATCH):
    wavs = np.stack([load_clip(p) for p in paths[k : k + BATCH]])
    with torch.no_grad():
        emb = model.get_audio_embedding_from_data(x=torch.from_numpy(wavs), use_tensor=True)
    audio_embs.append(torch.nn.functional.normalize(emb, dim=1).cpu())
audio_emb = torch.cat(audio_embs).numpy()
encode_sec = time.perf_counter() - t0
print(f"编码 {len(paths)} 片段耗时 {encode_sec:.1f} 秒(每片段 {encode_sec / len(paths) * 1000:.0f} ms)")


def encode_text(queries):
    with torch.no_grad():
        emb = model.get_text_embedding(queries, use_tensor=True)
    return torch.nn.functional.normalize(emb, dim=1).cpu().numpy()


def rank_table(query, top=10):
    # 一条文本查询对全库的余弦排序，返回（名次, clip_id, category, 相似度）
    q = encode_text([query])[0]
    sims = audio_emb @ q
    order = np.argsort(-sims)
    return [(r + 1, corpus.iloc[i]["clip_id"], corpus.iloc[i]["category"], float(sims[i])) for r, i in enumerate(order[:top])]


def first_rank(query, targets):
    # 目标集合中最佳名次（检索评估的基本量）
    q = encode_text([query])[0]
    sims = audio_emb @ q
    order = np.argsort(-sims)
    for r, i in enumerate(order, 1):
        if corpus.iloc[i]["category"] in targets:
            return r
    return len(order)


## 1. 三组查询的诊断性对照

三组查询检索同一个 190 片段的曲库，因此候选总数相同。但每条查询的目标集合大小不同：粗粒度“乐器”目标很多，细粒度乐器类通常只有少量片段。H@k 和 MRR 也会受到目标数量与构成影响，组间差异不等于查询粒度的单独作用。

粗粒度层查询乐器、人声和鼓。细粒度层查询胡琴家族中的二胡、板胡、中胡、高音板胡和壮剧土胡，用于观察近缘类别的排序。下方同时给出每条查询的最佳目标名次，以及分组后的 H@1、H@5 和 MRR。

In [ ]:
HUQIN_CLASSES = ["erhu", "banhu", "zhonghu", "gaoyin banhu", "tuhu"]
INSTRUMENT_CLASSES = sorted(corpus[corpus["source_group"].str.startswith("ctis") | (corpus["source_group"] == "folk_synth")]["category"].unique())

# 粗粒度层：乐器 / 人声 / 鼓
coarse_queries = [
    ("a musical instrument performance", set(INSTRUMENT_CLASSES)),
    ("isolated singing voice", {"vocals"}),
    ("drums only", {"drums"}),
]
coarse_rows = []
for query, targets in coarse_queries:
    r = first_rank(query, targets)
    coarse_rows.append({"查询": query, "目标类": "/".join(sorted(targets))[:28], "最佳名次": r, "H@1": r == 1, "H@5": r <= 5})
coarse_df = pd.DataFrame(coarse_rows)
print(coarse_df.to_string(index=False))

# 细粒度层：胡琴家族五类互查
fine_rows = []
for cls in HUQIN_CLASSES:
    query = f"a solo {cls} performance"
    r = first_rank(query, {cls})
    fine_rows.append({"查询": query, "目标类": cls, "最佳名次": r, "H@1": r == 1, "H@5": r <= 5})
fine_df = pd.DataFrame(fine_rows)
print(fine_df.to_string(index=False))

# 看一类查询的 Top-8，细粒度的串扰长什么样
print()
print("查询 'a solo erhu performance' 的 Top-8:")
print(pd.DataFrame(rank_table("a solo erhu performance", top=8), columns=["名次", "clip_id", "category", "相似度"]).to_string(index=False))


In [ ]:
# 属性级：快慢二胡，看属性词有没有把对应片段往前排
erhu_idx = corpus.index[corpus["category"] == "erhu"].tolist()
attr_cols = corpus.loc[erhu_idx, ["clip_id", "attribute"]].fillna("").reset_index(drop=True)
for query in ["a fast erhu performance", "a slow erhu performance", "a solo erhu performance"]:
    q = encode_text([query])[0]
    sims = audio_emb @ q
    order = np.argsort(-sims)
    rank_of = {int(i): int(r) for r, i in enumerate(order, 1)}
    attr_cols[query] = [round(float(sims[i]), 4) for i in erhu_idx]
    attr_cols[f"{query} 名次"] = [rank_of[i] for i in erhu_idx]
print(attr_cols.to_string(index=False))

# 类别查询汇总：H@1、H@5 与 MRR（属性层样本过少，单列说明）
layer_rows = [
    {"层": "粗粒度(乐器/人声/鼓)", "H@1": coarse_df["H@1"].mean(), "H@5": coarse_df["H@5"].mean(),
     "MRR": float(np.mean([1 / r for r in coarse_df["最佳名次"]]))},
    {"层": "细粒度(胡琴五类)", "H@1": fine_df["H@1"].mean(), "H@5": fine_df["H@5"].mean(),
     "MRR": float(np.mean([1 / r for r in fine_df["最佳名次"]]))},
]
layer_df = pd.DataFrame(layer_rows)
print(layer_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(len(layer_df))
w = 0.36
ax.bar(x - w / 2, layer_df["H@1"], width=w, color="0.25", label="H@1")
ax.bar(x + w / 2, layer_df["H@5"], width=w, color="0.6", label="H@5")
ax.set_xticks(x, layer_df["层"], fontsize=9)
ax.set_ylabel("命中率")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
finish_figure(fig, OUTPUT_FIGURES / "09_4_layer_rk.png")
plt.show()


## 2. 中英查询对照

本章使用的 LAION-CLAP 检查点对不同语言和提示措辞的适配程度，需要在目标任务上验证。第六章的中国乐器实验观察到部分英文类名优于对应中文提示，但这不表示英文查询对每个词都更好。下面用六组中英文查询比较同一目标类别的最佳名次。

In [ ]:
zh_en_pairs = [
    ("a solo erhu performance", "二胡独奏", "erhu"),
    ("a solo pipa performance", "琵琶独奏", "pipa"),
    ("a solo guzheng performance", "古筝独奏", "guzheng"),
    ("a solo suona performance", "唢呐独奏", "suona"),
    ("drums only", "鼓", "drums"),
    ("isolated singing voice", "人声歌唱", "vocals"),
]
zh_en_rows = []
for en, zh, target in zh_en_pairs:
    r_en = first_rank(en, {target})
    r_zh = first_rank(zh, {target})
    zh_en_rows.append({"目标类": target, "英文查询": en, "英文最佳名次": r_en, "中文查询": zh, "中文最佳名次": r_zh})
zh_en_df = pd.DataFrame(zh_en_rows)
print(zh_en_df.to_string(index=False))
better_en = int((zh_en_df["英文最佳名次"] < zh_en_df["中文最佳名次"]).sum())
print(f"英文名次更优 {better_en}/{len(zh_en_df)} 组")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
x = np.arange(len(zh_en_df))
w = 0.36
ax.bar(x - w / 2, zh_en_df["英文最佳名次"], width=w, color="0.25", label="英文查询")
ax.bar(x + w / 2, zh_en_df["中文最佳名次"], width=w, color="0.6", label="中文查询")
ax.set_xticks(x, zh_en_df["中文查询"], fontsize=9)
ax.set_ylabel("目标类最佳名次（越小越好）")
ax.set_yscale("log")
ax.legend(fontsize=8)
finish_figure(fig, OUTPUT_FIGURES / "09_4_zh_en.png")
plt.show()


六组结果没有一致方向。鼓、人声和古筝的英文查询名次更靠前；二胡、琵琶和唢呐的中文查询名次更靠前。英文提示中的 erhu、pipa、suona 是拼音词形，但这一本身不足以解释差异。
分词方式、多语言训练覆盖、提示模板和音频样本构成都可能影响名次。当前实验没有隔离这些因素，也只有六组查询，证据不足以确定分词是主要机制。
现有结果只支持一个较窄的结论：在当前检查点和曲库中，查询语言与目标最佳名次的关系因类别而异。面向中文场景部署前，应在目标词表、提示模板和目标曲库上分别验证。

## 3. 嵌入空间的投影与结果解释

主成分分析（PCA）把 190 条高维音频嵌入投影到二维，便于观察总体分布。二维图只保留方差最大的两个方向，无法完整保持高维空间中的距离和邻域关系。

图中的分组同时对应数据来源、录音条件和内容类型，因此来源之间的分离不等于语义类别结构，也不足以解释粗粒度查询为何取得满分。五类胡琴在二维投影中有较多重叠，这与细粒度混淆相容，但不证明它们在完整嵌入空间中不可分。

速度属性排序不稳定，可能与训练文案覆盖有关。
190 条向量可以直接扫描全部余弦相似度。曲库扩大后，精确扫描的计算量和访存量随候选数线性增长。当延迟或吞吐量不满足要求时，可使用 FAISS 或 HNSW 等近似最近邻索引先召回候选，再进行精排。


In [ ]:
# PCA 二维投影，按来源大组着色（灰度），看类的聚散
from sklearn.decomposition import PCA

proj = PCA(n_components=2).fit_transform(audio_emb)
groups = corpus["source_group"].unique().tolist()
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for k, grp in enumerate(groups):
    mask = (corpus["source_group"] == grp).to_numpy()
    ax.scatter(
        proj[mask, 0], proj[mask, 1],
        s=26, color=str(0.12 + 0.16 * k), marker="osd^v"[k % 5], label=grp, linewidths=0,
    )
ax.legend(fontsize=8, markerscale=1.4)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
finish_figure(fig, OUTPUT_FIGURES / "09_4_embedding_projection.png")
plt.show()

# 胡琴五类在投影里的位置（单独放大）
fig, ax = plt.subplots(figsize=(7.5, 5.2))
huqin_mask = corpus["source_group"].isin(["ctis_huqin"]).to_numpy()
other_mask = ~huqin_mask
ax.scatter(proj[other_mask, 0], proj[other_mask, 1], s=12, color="0.8", linewidths=0)
for k, cls in enumerate(HUQIN_CLASSES):
    mask = (corpus["category"] == cls).to_numpy()
    ax.scatter(proj[mask, 0], proj[mask, 1], s=40, color=str(0.1 + 0.18 * k), marker="osd^v"[k % 5], label=cls, linewidths=0)
ax.legend(fontsize=8, markerscale=1.4)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
finish_figure(fig, OUTPUT_FIGURES / "09_4_huqin_projection.png")
plt.show()


## 4. 数据来源与评估方法

本章使用项目内已有音频构建小型语料。MusicCaps 是公开的音频文本标注数据之一，含 5521 个十秒音乐片段的英文描述和属性列表，片段来源于 AudioSet。公开数据文件主要提供标注、YouTube ID 和时间位置，不直接包含音频。

Kaggle 页面把 MusicCaps 数据文件标为 CC BY-SA 4.0，但这不等于对上游 YouTube 音频授予相同权利。复现实验还会受到视频可用性、媒体权利和平台访问条件影响，应分别核对。

本节报告 H@k 与 MRR，并同时给出查询数、目标类别和具体名次。

In [ ]:
coarse_df.to_csv(OUTPUT_TABLES / "09_4_coarse_layer.csv", index=False)
fine_df.to_csv(OUTPUT_TABLES / "09_4_fine_layer.csv", index=False)
layer_df.to_csv(OUTPUT_TABLES / "09_4_layer_summary.csv", index=False)
attr_cols.to_csv(OUTPUT_TABLES / "09_4_attribute_layer.csv", index=False)
zh_en_df.to_csv(OUTPUT_TABLES / "09_4_zh_en.csv", index=False)
print("表格已写入", rel(OUTPUT_TABLES))
print("图已写入", rel(OUTPUT_FIGURES))
